# The Movies We Remember
## A visual exploration of IMDb's top 1,000 films

**Exploratory data analysis in R** | ratings, genres, eras, audience attention, and box office

This notebook explores a curated film dataset: what gets rated highly, which genres appear, how release eras are represented, and how audience attention relates to reported revenue. Statistics and graphics are calculated from the CSV when the notebook runs.

> Keep this notebook beside `movie.csv`, install R with the Jupyter `IRkernel`, and select an R kernel in VS Code. If the CSV is a Git LFS pointer rather than the actual data, run `git lfs pull` first.

## 1. Setup
We use `tidyverse` for data work and plotting, plus `scales` and `knitr` for readable axes and tables. The palette pairs deep teal with warm coral and gold for a consistent visual identity.

In [1]:
# Kaggle's R image usually includes these packages. Install only when needed;
# this also keeps the notebook usable in a fresh local IRkernel environment.
required_packages <- c("tidyverse", "scales", "knitr")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages)) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org", quiet = TRUE)
}
invisible(lapply(required_packages, library, character.only = TRUE))
options(scipen = 999, dplyr.width = Inf)

theme_set(theme_minimal(base_size = 12, base_family = "sans"))
ink <- "#18323A"
coral <- "#D65A4A"
teal <- "#168C86"
gold <- "#E5A83B"
blue <- "#4C78A8"
plot_theme <- theme(
  plot.title = element_text(face = "bold", size = 15, color = ink),
  plot.subtitle = element_text(color = "#52666B"),
  axis.title = element_text(face = "bold", color = ink),
  panel.grid.minor = element_blank(),
  plot.title.position = "plot",
  legend.position = "bottom"
)
cat("R version:", R.version.string, "\nPackages ready:", paste(required_packages, collapse = ", "))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: 'scales'


The following object is masked from 'package:purrr':

    discard


The following object is masked from 'package:readr':

    col_factor




R version: R version 4.6.1 (2026-06-24 ucrt) 
Packages ready: tidyverse, scales, knitr

## 2. Load and inspect
Attach the IMDb movie CSV through Kaggle?s **Add Input** panel. The loader searches attached CSV files by their expected columns, so the uploaded filename does not need to be `movie.csv`.


In [2]:
# Kaggle mounts attached datasets under /kaggle/input/<dataset-slug>/.
# Discover CSV files by their expected schema, so the uploaded filename can vary.
required_columns <- c("Series_Title", "Released_Year", "Certificate", "Runtime", "Genre",
                      "IMDB_Rating", "Meta_score", "Director", "Star1", "No_of_Votes", "Gross")
candidate_paths <- unique(c(
  "D:/Assignment_SMIT_AIDS_2026-main/Assignment_4(EDA)/movie.csv",
  if (dir.exists("/kaggle/input")) list.files("/kaggle/input", pattern = "\\.csv$", recursive = TRUE, full.names = TRUE, ignore.case = TRUE) else character(),
  list.files(".", pattern = "\\.csv$", full.names = TRUE, ignore.case = TRUE),
  list.files("..", pattern = "\\.csv$", full.names = TRUE, ignore.case = TRUE)
))
candidate_paths <- candidate_paths[file.exists(candidate_paths)]
if (!length(candidate_paths)) {
  stop("No CSV file found. Attach the movie dataset in Kaggle's Input panel or put its CSV beside this notebook.")
}
# Ignore Git LFS pointer stubs and identify the dataset by its columns, not its filename.
usable_paths <- character()
for (path in candidate_paths) {
  first_line <- readLines(path, n = 1, warn = FALSE)
  if (!length(first_line) || grepl("git-lfs.github.com/spec/v1", first_line)) next
  header <- tryCatch(names(readr::read_csv(path, n_max = 0, show_col_types = FALSE,
                                           name_repair = "minimal")), error = function(e) character())
  if (all(required_columns %in% header)) usable_paths <- c(usable_paths, path)
}
if (!length(usable_paths)) {
  stop(paste0("No usable movie CSV found. The local movie.csv files are Git LFS pointers, ",
              "not the dataset. Attach the actual IMDb Top 1000 CSV in Kaggle's Input panel; ",
              "the filename can be anything as long as it has the expected movie columns."))
}
csv_path <- usable_paths[[1]]
movies_raw <- readr::read_csv(csv_path, show_col_types = FALSE,
                              na = c("", "NA", "N/A", "Not Available"),
                              name_repair = "minimal")
cat("Loaded:", csv_path, "| Rows:", nrow(movies_raw), "| Columns:", ncol(movies_raw), "\n")
knitr::kable(head(movies_raw, 6), caption = "First six records")

ERROR: Error: No usable movie CSV found. The local movie.csv files are Git LFS pointers, not the dataset. Attach the actual IMDb Top 1000 CSV in Kaggle's Input panel; the filename can be anything as long as it has the expected movie columns.


In [ ]:
missing_columns <- setdiff(required_columns, names(movies_raw))
if (length(missing_columns)) stop("Expected columns missing: ", paste(missing_columns, collapse = ", "))
movies <- movies_raw %>%
  mutate(across(where(is.character), ~na_if(trimws(.x), "")),
         Released_Year = readr::parse_number(as.character(Released_Year)),
         Runtime_min = readr::parse_number(as.character(Runtime)),
         IMDB_Rating = readr::parse_number(as.character(IMDB_Rating)),
         Meta_score = readr::parse_number(as.character(Meta_score)),
         No_of_Votes = readr::parse_number(as.character(No_of_Votes)),
         Gross = readr::parse_number(as.character(Gross)),
         Decade = floor(Released_Year / 10) * 10,
         Rating_band = cut(IMDB_Rating, breaks = c(0, 7, 8, 9, 10),
                           labels = c("Below 7", "7 to <8", "8 to <9", "9+"), include.lowest = TRUE))
genres <- movies %>% select(Series_Title, Released_Year, IMDB_Rating, No_of_Votes, Gross, Genre) %>%
  filter(!is.na(Genre)) %>% separate_rows(Genre, sep = ",\\s*") %>% mutate(Genre = trimws(Genre))
glimpse(movies)
cat("Expanded film-genre rows:", nrow(genres))

## 3. Data quality and a first visual
We keep incomplete rows and avoid filling ratings or revenue with guessed values. Each comparison uses the available observations for its fields.

In [ ]:
missing_summary <- tibble(
  column = names(movies_raw),
  missing_n = vapply(movies_raw, function(x) sum(is.na(x)), integer(1)),
  missing_pct = 100 * missing_n / nrow(movies_raw)
) %>% arrange(desc(missing_pct))
knitr::kable(missing_summary, digits = 1, caption = "Missing values by field")
ggplot(missing_summary, aes(reorder(column, missing_pct), missing_pct)) +
  geom_col(fill = teal, width = 0.72) + coord_flip() +
  geom_text(aes(label = paste0(round(missing_pct, 1), "%")), hjust = -0.12, size = 3.4, color = ink) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.14))) +
  labs(title = "Some fields are much less complete than others",
       subtitle = "Share of records with missing values", x = NULL, y = "Missing rows (%)") + plot_theme

In [ ]:
numeric_summary <- tibble(
  field = c("Release year", "Runtime (min)", "IMDb rating", "Metascore", "Votes", "Reported gross"),
  available_n = c(sum(!is.na(movies$Released_Year)), sum(!is.na(movies$Runtime_min)), sum(!is.na(movies$IMDB_Rating)), sum(!is.na(movies$Meta_score)), sum(!is.na(movies$No_of_Votes)), sum(!is.na(movies$Gross))),
  median = c(median(movies$Released_Year, na.rm = TRUE), median(movies$Runtime_min, na.rm = TRUE), median(movies$IMDB_Rating, na.rm = TRUE), median(movies$Meta_score, na.rm = TRUE), median(movies$No_of_Votes, na.rm = TRUE), median(movies$Gross, na.rm = TRUE))
)
knitr::kable(numeric_summary, digits = 2, caption = "Numeric fields: availability and median")
ggplot(movies, aes(IMDB_Rating)) + geom_histogram(binwidth = 0.1, fill = coral, color = "white") +
  geom_vline(xintercept = median(movies$IMDB_Rating, na.rm = TRUE), linetype = 2, color = ink) +
  labs(title = "IMDb ratings cluster in a narrow upper range", x = "IMDb rating", y = "Films") + plot_theme
ggplot(movies, aes(Runtime_min)) + geom_histogram(binwidth = 10, fill = teal, color = "white") +
  labs(title = "Film runtimes span compact features to epics", x = "Runtime (minutes)", y = "Films") + plot_theme
ggplot(movies, aes(x = "All films", y = Runtime_min)) +
  geom_boxplot(fill = gold, alpha = 0.85, width = 0.35, outlier.color = coral) +
  labs(title = "Runtime outliers", x = NULL, y = "Runtime (minutes)") + plot_theme +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())

## 4. Film history
This curated sample is not a census of every movie released. The charts describe which decades are represented here and how their median audience ratings compare.

In [ ]:
decade_counts <- movies %>% filter(!is.na(Decade)) %>% count(Decade)
ggplot(decade_counts, aes(Decade, n)) + geom_col(fill = blue, width = 8) +
  scale_x_continuous(breaks = decade_counts$Decade, labels = paste0(decade_counts$Decade, "s")) +
  labs(title = "Which eras appear most in the dataset?", x = "Release decade", y = "Films") + plot_theme +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
decade_ratings <- movies %>% filter(!is.na(Decade), !is.na(IMDB_Rating)) %>%
  group_by(Decade) %>% summarise(median_rating = median(IMDB_Rating), films = n(), .groups = "drop")
ggplot(decade_ratings, aes(Decade, median_rating, size = films)) +
  geom_line(color = teal, linewidth = 1) + geom_point(color = coral, alpha = 0.9) +
  scale_size_continuous(name = "Films", range = c(2, 8)) +
  labs(title = "Median IMDb rating by release decade", x = "Release decade", y = "Median rating") + plot_theme

## 5. Genres and certification
Genre labels overlap: a single movie can contribute to several genre counts. Ratings by genre therefore compare film-genre appearances, not mutually exclusive groups.

In [ ]:
genre_counts <- genres %>% count(Genre, sort = TRUE) %>% slice_max(n, n = 12, with_ties = FALSE)
ggplot(genre_counts, aes(reorder(Genre, n), n, fill = n)) + geom_col(show.legend = FALSE) + coord_flip() +
  scale_fill_gradient(low = "#B8DAD5", high = teal) +
  labs(title = "Drama and its neighbors", subtitle = "12 most frequent genre labels", x = NULL, y = "Film-genre appearances") + plot_theme
selected_genres <- genre_counts %>% slice_max(n, n = 8, with_ties = FALSE) %>% pull(Genre)
genre_ratings <- genres %>% filter(Genre %in% selected_genres, !is.na(IMDB_Rating)) %>%
  mutate(Genre = forcats::fct_reorder(Genre, IMDB_Rating, median))
ggplot(genre_ratings, aes(Genre, IMDB_Rating, fill = Genre)) + geom_boxplot(show.legend = FALSE, outlier.alpha = 0.25) + coord_flip() +
  scale_fill_manual(values = rep(c(coral, teal, gold, blue), 2)) +
  labs(title = "Ratings vary across common genres", x = NULL, y = "IMDb rating") + plot_theme
certificate_counts <- movies %>% filter(!is.na(Certificate)) %>% count(Certificate, sort = TRUE) %>% slice_max(n, n = 10, with_ties = FALSE)
ggplot(certificate_counts, aes(reorder(Certificate, n), n)) + geom_col(fill = gold) + coord_flip() +
  labs(title = "Certification labels in the sample", x = NULL, y = "Films") + plot_theme

## 6. Audience attention and critical scores
Vote counts and gross are right-skewed, so logarithmic axes make small and large values legible together. Trend lines describe association, not causation.

In [ ]:
ggplot(movies %>% filter(!is.na(No_of_Votes), No_of_Votes > 0), aes(No_of_Votes, IMDB_Rating)) +
  geom_point(color = teal, alpha = 0.55, size = 2) + geom_smooth(method = "lm", se = TRUE, color = coral) +
  scale_x_log10(labels = scales::label_number(big.mark = ",")) +
  labs(title = "Highly rated films also tend to attract votes", x = "IMDb votes (log scale)", y = "IMDb rating") + plot_theme
ggplot(movies %>% filter(!is.na(Gross), Gross > 0), aes(Gross, IMDB_Rating)) +
  geom_point(color = coral, alpha = 0.55, size = 2) + geom_smooth(method = "lm", se = TRUE, color = ink) +
  scale_x_log10(labels = scales::label_dollar(scale = 1e-6, suffix = "M")) +
  labs(title = "Reported gross is not a rating shortcut", x = "Gross (USD millions; log scale)", y = "IMDb rating") + plot_theme
ggplot(movies %>% filter(!is.na(Gross), Gross > 0), aes(Gross)) +
  geom_histogram(bins = 35, fill = gold, color = "white") +
  scale_x_log10(labels = scales::label_dollar(scale = 1e-6, suffix = "M")) +
  labs(title = "Reported box office has a long upper tail", x = "Gross (USD millions; log scale)", y = "Films") + plot_theme
ggplot(movies %>% filter(!is.na(Meta_score), !is.na(IMDB_Rating)), aes(Meta_score, IMDB_Rating)) +
  geom_point(color = blue, alpha = 0.55, size = 2) + geom_smooth(method = "lm", se = TRUE, color = coral) +
  labs(title = "Critic and audience scores move together, imperfectly", x = "Metascore", y = "IMDb rating") + plot_theme
ggplot(movies %>% filter(!is.na(Runtime_min), !is.na(IMDB_Rating)), aes(Runtime_min, IMDB_Rating)) +
  geom_point(color = teal, alpha = 0.5, size = 2) + geom_smooth(method = "loess", se = TRUE, color = coral) +
  labs(title = "Does runtime trace a clear rating pattern?", x = "Runtime (minutes)", y = "IMDb rating") + plot_theme

## 7. Genre economics and filmmakers
Revenue is available for only some titles and spans many decades. These are descriptive comparisons: gross is not adjusted for inflation, ticket prices, or release scale.

In [ ]:
genre_attention <- genres %>% filter(!is.na(No_of_Votes)) %>% group_by(Genre) %>%
  summarise(median_votes = median(No_of_Votes), films = n(), .groups = "drop") %>% filter(films >= 10) %>% slice_max(median_votes, n = 12)
ggplot(genre_attention, aes(reorder(Genre, median_votes), median_votes)) + geom_col(fill = blue) + coord_flip() +
  scale_y_log10(labels = scales::label_number(big.mark = ",")) +
  labs(title = "Audience attention by genre", x = NULL, y = "Median votes (log scale)") + plot_theme
genre_gross <- genres %>% filter(!is.na(Gross), Gross > 0) %>% group_by(Genre) %>%
  summarise(median_gross = median(Gross), films = n(), .groups = "drop") %>% filter(films >= 10) %>% slice_max(median_gross, n = 12)
ggplot(genre_gross, aes(reorder(Genre, median_gross), median_gross)) + geom_col(fill = coral) + coord_flip() +
  scale_y_continuous(labels = scales::label_dollar(scale = 1e-6, suffix = "M")) +
  labs(title = "Median reported gross among common genres", x = NULL, y = "Gross (USD millions)") + plot_theme
director_counts <- movies %>% filter(!is.na(Director)) %>% count(Director, sort = TRUE) %>% slice_max(n, n = 15)
ggplot(director_counts, aes(reorder(Director, n), n)) + geom_col(fill = teal) + coord_flip() +
  labs(title = "Directors with the most entries", x = NULL, y = "Films in dataset") + plot_theme
top_gross <- movies %>% filter(!is.na(Gross), Gross > 0) %>% slice_max(Gross, n = 12) %>%
  mutate(Series_Title = forcats::fct_reorder(Series_Title, Gross))
ggplot(top_gross, aes(Series_Title, Gross, fill = IMDB_Rating)) + geom_col() + coord_flip() +
  scale_fill_gradient(low = gold, high = teal, name = "IMDb rating") +
  scale_y_continuous(labels = scales::label_dollar(scale = 1e-6, suffix = "M")) +
  labs(title = "Highest reported gross in the dataset", x = NULL, y = "Gross (USD millions)") + plot_theme

## 8. Relationships and takeaways
The correlation heatmap compares numeric fields using pairwise-complete observations. These values describe association only; the closing summary is calculated from the loaded file.

In [ ]:
numeric_fields <- movies %>% select(Released_Year, Runtime_min, IMDB_Rating, Meta_score, No_of_Votes, Gross)
correlation_matrix <- cor(numeric_fields, use = "pairwise.complete.obs")
correlation_long <- as.data.frame(as.table(correlation_matrix)) %>% rename(field_x = Var1, field_y = Var2, correlation = Freq)
ggplot(correlation_long, aes(field_x, field_y, fill = correlation)) +
  geom_tile(color = "white", linewidth = 0.7) + geom_text(aes(label = sprintf("%.2f", correlation)), color = ink, size = 3.6) +
  scale_fill_gradient2(low = blue, mid = "#F5F3EC", high = coral, midpoint = 0, limits = c(-1, 1)) + coord_equal() +
  labs(title = "A compact map of numeric relationships", x = NULL, y = NULL, fill = "Correlation") + plot_theme +
  theme(axis.text.x = element_text(angle = 35, hjust = 1))
top_voted <- movies %>% filter(!is.na(No_of_Votes)) %>% slice_max(No_of_Votes, n = 10) %>%
  select(Series_Title, Released_Year, IMDB_Rating, No_of_Votes, Gross) %>% arrange(desc(No_of_Votes))
knitr::kable(top_voted, digits = 1, caption = "Ten most-voted titles")
takeaways <- tibble(
  question = c("Titles represented", "Median IMDb rating", "Median runtime", "Most common genre label", "Records missing reported gross"),
  result = c(as.character(nrow(movies)), sprintf("%.2f / 10", median(movies$IMDB_Rating, na.rm = TRUE)),
             sprintf("%.0f minutes", median(movies$Runtime_min, na.rm = TRUE)),
             as.character(names(sort(table(genres$Genre), decreasing = TRUE))[1]), as.character(sum(is.na(movies$Gross))))
)
knitr::kable(takeaways, col.names = c("Question", "Computed from this file"), caption = "Reproducible headline facts")

## Conclusion
These views describe patterns inside this particular curated dataset, not universal laws about movies. Rankings favor acclaimed and visible films; reported revenue is incomplete and unadjusted; genre labels overlap. Record your own conclusions after reviewing the charts and computed summaries.

**Reproducibility:** source data remain unchanged. Parsed fields and genre expansions are created in memory when the notebook runs.